# Lista de Exercícios 2 - Advanced Econometrics (2026)

Mestrado Profissional em Economia, Insper. Prof. Hedibert Freitas Lopes. Due 23/11/2026, 19h30.

**Group:** (names)

**Data.** $n = 40$ adults from Alachua County, Florida (Agresti et al. 2026, §4.1.5, file `Mental.dat`): $y_i$ = `impair` (mental impairment index), $x_{1i}$ = `life` (life-events index), $x_{2i}$ = `ses` (socioeconomic status, 0–100).

**Models.** $z_{ji} = (x_{ji}-\bar x_j)/s_{x_j}$. M1: $y_i=\beta_0+\beta_1z_{1i}+\varepsilon_i$; M2: $y_i=\beta_0+\beta_2z_{2i}+\varepsilon_i$; M3: both regressors; $\varepsilon_i\sim N(0,\sigma^2)$. M4–M6: the same three mean functions with $\varepsilon_i\sim t_\nu(0,\sigma^2)$, $\nu$ unknown. Priors (independent, common to all models): $\beta_0\sim N(25,5^2)$, $\beta_j\sim N(0,3^2)$, $\sigma^2\sim IG(3,50)$, $\nu\sim\text{Gamma}(2,0.1)$ (shape, rate).

**How to read this report.** Each item opens with its statement, quoted verbatim (in Portuguese) from the assignment, right before its code and answer. Each item has an "Answer" cell whose first line is the result; the derivation or discussion follows, and every number quoted comes from the output just above it. All computations use Python (numpy, scipy, pandas, matplotlib). The samplers, the effective-size and $\hat R$ diagnostics, the marginal-likelihood estimators and PSIS are written out in this notebook (no PPL), with a fixed seed. Iterations, burn-in and effective sizes are reported in 2(b) and in the final section.

## Setup and data

The data are typed in the file order of page 2 of the statement. The check table compares them with the summaries printed in the statement.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize, integrate
from scipy.special import gammaln, logsumexp

plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.precision", 3)
SEED = 2026        # master seed: every sampler call below derives its own seed from it
T0 = time.time()

In [ ]:
impair = np.array([17,19,20,20,20,21,21,22,22,23,24,24,25,26,26,26,26,27,27,27,
                   27,28,28,28,28,28,29,30,30,31,31,31,31,32,33,34,34,34,41,41])
life = np.array([46,39,27, 3,10,44,37,35,78,32,33,18,81,22,50,48,45,21,55,45,
                 60,97,37,30,13,40, 5,59,44,35,95,63,42,38,45,70,57,40,49,89])
ses = np.array([84,97,24,85,15,55,78,91,60,74,67,39,87,95,40,52,61,45,88,56,
                70,89,50,90,56,56,40,72,53,38,29,53, 7,32,55,58,16,29, 3,75])
Mental = pd.DataFrame({"impair": impair, "life": life, "ses": ses})
n = len(Mental)
y = impair.astype(float)
z1 = (life - life.mean()) / life.std(ddof=1)
z2 = (ses - ses.mean()) / ses.std(ddof=1)

check = pd.DataFrame({"mean": Mental.mean(), "s.d.": Mental.std(ddof=1),
                      "min": Mental.min(), "max": Mental.max(),
                      "statement mean": [27.30, 44.42, 56.60],
                      "statement s.d.": [5.46, 22.62, 25.28]})
check.round(2)

---
## Question 1: Análise exploratória e correlações

#### Statement 1(a)

> **(a)** Reporte média, mediana, desvio-padrão, mínimo e máximo de $y$, $x_1$ e $x_2$, e faça os histogramas das três variáveis.

In [ ]:
desc = Mental.agg(["mean", "median", "std", "min", "max"]).T.rename(columns={"std": "s.d."})
display(desc.round(2))

labels = {"impair": "y = impair (mental impairment)", "life": "x1 = life (life events)",
          "ses": "x2 = ses (socioeconomic status)"}
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for a, (c, lab) in zip(ax, labels.items()):
    a.hist(Mental[c], bins=8, edgecolor="white")
    a.axvline(Mental[c].mean(), color="C3", lw=2, label=f"mean {Mental[c].mean():.1f}")
    a.axvline(Mental[c].median(), color="k", ls="--", lw=1.5, label=f"median {Mental[c].median():.0f}")
    a.set_xlabel(lab); a.set_ylabel("count"); a.legend(fontsize=8)
fig.suptitle("Figure 1. Histograms of the three variables (n = 40)")
fig.tight_layout(); plt.show()

In [ ]:
# Shape and atypical values: skewness and the 1.5*IQR rule (Tukey fences)
rows = {}
for c in Mental:
    q1, q3 = np.percentile(Mental[c], [25, 75])
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    out = Mental.index[(Mental[c] < lo) | (Mental[c] > hi)] + 1
    rows[c] = {"skewness": stats.skew(Mental[c]), "Q1": q1, "Q3": q3,
               "lower fence": lo, "upper fence": hi, "obs. outside (i)": list(out)}
pd.DataFrame(rows).T

### Answer 1(a)

**Result:**

| | mean | median | s.d. | min | max |
|---|---|---|---|---|---|
| $y$ = impair | 27.30 | 27 | 5.46 | 17 | 41 |
| $x_1$ = life | 44.42 | 43 | 22.62 | 3 | 97 |
| $x_2$ = ses | 56.60 | 56 | 25.28 | 3 | 97 |

The typed data reproduce the summaries in the statement (check table in the setup). Standard deviations use the denominator $n-1$.

Reading Figure 1:

- **$y$ (impair).** One peak at 26 to 29 (13 of the 40 people) and a mild right skew (skewness 0.44; mean 27.3 just above the median 27). The two largest values, $y=41$ (observations 39 and 40), are separated from the rest by an empty bin: no one scores between 35 and 40. They still lie inside the Tukey fences (upper fence $Q_3+1.5\,IQR = 41.9$), so the 1.5·IQR rule does not flag them as outliers.
- **$x_1$ (life).** Right-skewed (skewness 0.50). About half (21 people) are between 27 and 50, with a long right tail. Observations 22 (life = 97) and 31 (life = 95) lie beyond the upper fence of 89.6.
- **$x_2$ (ses).** Values spread over the whole 0 to 100 scale, with a slight left skew (skewness −0.26): 7 people above 85 and a thinner lower tail. No value lies outside the fences.

#### Statement 1(b)

> **(b)** Faça os diagramas de dispersão $y\times x_1$, $y\times x_2$ e $x_1\times x_2$ (por exemplo, `pairs(Mental)`).

In [ ]:
pairs = [("life", "impair"), ("ses", "impair"), ("life", "ses")]
flag = (Mental.impair <= 17) | (Mental.impair >= 41) | (Mental.life >= 90) | (Mental.ses <= 7)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
for a, (xc, yc) in zip(ax, pairs):
    a.scatter(Mental[xc], Mental[yc], s=18)
    b1, b0 = np.polyfit(Mental[xc], Mental[yc], 1)
    g = np.linspace(Mental[xc].min(), Mental[xc].max(), 2)
    a.plot(g, b0 + b1 * g, color="C3", lw=1.5, label="least-squares line")
    for i in Mental.index[flag]:
        a.annotate(str(i + 1), (Mental[xc][i], Mental[yc][i]), fontsize=7,
                   xytext=(3, 3), textcoords="offset points")
    a.set_xlabel(xc); a.set_ylabel(yc); a.legend(fontsize=8)
fig.suptitle("Figure 2. Scatter plots (numbers = observation index i, file order)")
fig.tight_layout(); plt.show()

### Answer 1(b)

**Result:** impairment increases with life events and decreases with SES, both with wide scatter around the line. Life events and SES show no clear relation.

- **$y \times x_1$:** positive slope. The relation looks linear, with no visible curvature, and the vertical spread looks similar along the axis. Observations 22 and 31 (the two extreme life values) lie slightly below the line, so they do not drive the slope.
- **$y \times x_2$:** negative slope, linear, similar spread. Observation 40 is the clearest point off the line: very high impairment (41) despite a high SES (75). Observation 39 (impairment 41, SES 3, the lowest in the sample) follows the negative trend.
- **$x_1 \times x_2$:** a cloud with a nearly flat least-squares line. The two regressors carry nearly separate information.

The points numbered in the figure come back in Question 3(c).

#### Statement 1(c)

> **(c)** Calcule as correlações de Pearson $r(y,x_1)$, $r(y,x_2)$ e $r(x_1,x_2)$. Para cada uma, construa um intervalo de confiança de 95% (via transformação $z$ de Fisher) e teste $H_0:\rho=0$. Compare também com a correlação de Spearman.

In [ ]:
def corr_table(df, pairs, level=0.95):
    """Pearson r with Fisher-z CI, t test of H0: rho = 0, and Spearman r_s for each pair."""
    zq = stats.norm.ppf(0.5 + level / 2)
    rows = {}
    for a, b in pairs:
        r = df[a].corr(df[b])
        z, se = np.arctanh(r), 1 / np.sqrt(len(df) - 3)
        t_stat = r * np.sqrt(len(df) - 2) / np.sqrt(1 - r**2)       # ~ t_{n-2} under H0
        rs = stats.spearmanr(df[a], df[b])
        rows[f"r({a}, {b})"] = {
            "Pearson r": r, "CI 2.5%": np.tanh(z - zq * se), "CI 97.5%": np.tanh(z + zq * se),
            "t stat (df=38)": t_stat, "p-value": 2 * stats.t.sf(abs(t_stat), len(df) - 2),
            "Fisher z stat": z / se, "p-value (z)": 2 * stats.norm.sf(abs(z / se)),
            "Spearman r_s": rs.statistic, "p-value (r_s)": rs.pvalue}
    return pd.DataFrame(rows).T

corr = corr_table(Mental, [("impair", "life"), ("impair", "ses"), ("life", "ses")])
corr.round(4)

### Answer 1(c)

**Result:**

| pair | Pearson $r$ | 95% CI (Fisher $z$) | $t$ stat (38 df) | p-value | Spearman $r_s$ (p-value) |
|---|---|---|---|---|---|
| $(y, x_1)$ | 0.372 | [0.069, 0.613] | 2.472 | 0.018 | 0.368 (0.019) |
| $(y, x_2)$ | −0.399 | [−0.632, −0.099] | −2.679 | 0.011 | −0.402 (0.010) |
| $(x_1, x_2)$ | 0.123 | [−0.196, 0.419] | 0.766 | 0.448 | 0.088 (0.590) |

**Interval.** $z = \operatorname{artanh}(r) = \tfrac12\log\frac{1+r}{1-r}$ is approximately $N\!\left(\operatorname{artanh}\rho,\ \tfrac{1}{n-3}\right)$. The interval is $\tanh\!\left(z \pm 1.96/\sqrt{37}\right)$. For $(y,x_1)$: $z = 0.391$, $0.391 \pm 1.96 \times 0.164 = [0.069, 0.713]$ on the $z$ scale, which $\tanh$ maps back to $[0.069, 0.613]$.

**Test of $H_0:\rho=0$ against $H_1:\rho\neq0$.** Statistic $t = r\sqrt{n-2}/\sqrt{1-r^2}$, which follows a $t_{38}$ distribution under $H_0$ for bivariate normal data. The Fisher-$z$ version, $z\sqrt{n-3}$ compared with $N(0,1)$, gives 2.378 (p-value 0.017), −2.567 (0.010) and 0.754 (0.451): the same decisions.

- $(y,x_1)$: $t = 2.472$, p-value 0.018. Reject $H_0$ at 5% (not at 1%).
- $(y,x_2)$: $t = -2.679$, p-value 0.011. Reject $H_0$ at 5%.
- $(x_1,x_2)$: $t = 0.766$, p-value 0.448. Do not reject $H_0$; the interval contains 0.

**Spearman.** Same signs and the same decisions. For the two pairs involving $y$, Spearman is almost equal to Pearson (0.368 vs 0.372; −0.402 vs −0.399). Since Spearman works on ranks, which ignore the size of extreme values, the two associations are monotone and are not produced by a few extreme points such as observations 22 and 31. For $(x_1,x_2)$, Spearman is lower (0.088 vs 0.123), so part of the small Pearson value comes from extreme points.

With $n=40$ the intervals are wide: the data on $(y,x_1)$ are compatible with anything from a negligible correlation (0.07) to a strong one (0.61).

#### Statement 1(d)

> **(d)** Interprete sinais e magnitudes. A correlação entre as duas variáveis explicativas é forte o suficiente para gerar preocupação com multicolinearidade? O que você espera, a partir das correlações, sobre os coeficientes do modelo com ambas as variáveis quando comparados aos dos modelos com uma variável só?

In [ ]:
r = Mental.corr()
r12, ry1, ry2 = r.loc["life", "ses"], r.loc["impair", "life"], r.loc["impair", "ses"]
sy = Mental.impair.std(ddof=1)
print(f"VIF = 1/(1 - r12^2) = {1 / (1 - r12**2):.3f}")
# Coefficients in index points per s.d. of x implied by the correlations (least squares):
print(f"one regressor : beta1 = r_y1*s_y = {ry1 * sy:.2f}, beta2 = r_y2*s_y = {ry2 * sy:.2f}")
b1 = (ry1 - ry2 * r12) / (1 - r12**2) * sy
b2 = (ry2 - ry1 * r12) / (1 - r12**2) * sy
print(f"both          : beta1 = {b1:.2f}, beta2 = {b2:.2f}")
print(f"R^2: life only {ry1**2:.3f}, ses only {ry2**2:.3f}, both "
      f"{(ry1**2 + ry2**2 - 2 * ry1 * ry2 * r12) / (1 - r12**2):.3f}")

### Answer 1(d)

**Result:** both associations are moderate and point the expected way. Collinearity is negligible ($r_{12}=0.12$, VIF = 1.015). In the model with both variables, each coefficient should be slightly larger in absolute value than in the one-regressor models: about $2.34$ and $-2.46$ instead of $2.03$ and $-2.18$.

- **Signs.** More life events go with more impairment ($r=0.37$). Higher socioeconomic status goes with less impairment ($r=-0.40$). These are associations in observational data from 40 people, not causal effects.
- **Magnitudes.** Each variable alone explains 14% ($r^2=0.139$) and 16% ($0.159$) of the variance of $y$. On the scale of the models, the least-squares slope of $y$ on $z_j$ is $r_{yj}s_y$: one s.d. more life events goes with 2.03 more points of impairment, and one s.d. more SES with 2.18 fewer points. The s.d. of $y$ is 5.46.
- **Multicollinearity.** $r(x_1,x_2)=0.123$ is small and not significant (p-value 0.448). The variance inflation factor $1/(1-r_{12}^2)=1.015$ means the variance of each coefficient in the two-regressor model grows by only 1.5% because of the correlation between the regressors. This is no concern: the usual warning levels are a VIF of 5 or 10.
- **What to expect in M3/M6.** With standardized variables, the least-squares coefficients are
$$\beta_1^{*}=\frac{r_{y1}-r_{y2}r_{12}}{1-r_{12}^2},\qquad \beta_2^{*}=\frac{r_{y2}-r_{y1}r_{12}}{1-r_{12}^2}.$$
Because $r_{12}>0$ while the two effects have opposite signs, each one-regressor model absorbs part of the opposite effect of the omitted variable (omitted-variable bias toward zero): $r_{y1} = \beta_1^* + \beta_2^* r_{12}$. The correlations imply slopes of 2.34 and −2.46 index points per s.d. in the joint model. That is 13% to 15% larger in absolute value than the single-variable slopes, with almost no change in their standard errors (VIF ≈ 1). Together the two variables explain $R^2=0.339$, more than the sum of the separate $R^2$ (0.298), so the residual s.d. should fall noticeably. Question 2(b) confirms both predictions.

---
## Question 2: Comparação de modelos via fator de Bayes

#### Statement 2(a)

> **(a)** Escreva a verossimilhança dos modelos Gaussianos (M1–M3) e dos modelos Student-t (M4–M6). Para estes últimos, escreva também a verossimilhança aumentada pelas variáveis latentes $\omega_1,\dots,\omega_n$.

### Answer 2(a)

Notation: $x_i$ is the row of the design matrix ($x_i=(1,z_{1i})'$ in M1/M4, $(1,z_{2i})'$ in M2/M5, $(1,z_{1i},z_{2i})'$ in M3/M6), $\beta$ the matching coefficient vector, and $e_i = y_i - x_i'\beta$. The observations are independent given the parameters, so each likelihood is a product.

**Gaussian models M1–M3.** With $y_i\mid\beta,\sigma^2 \sim N(x_i'\beta,\sigma^2)$:
$$L(\beta,\sigma^2;y)=\prod_{i=1}^{n}\frac{1}{\sqrt{2\pi\sigma^2}}\exp\left\{-\frac{(y_i-x_i'\beta)^2}{2\sigma^2}\right\}=(2\pi\sigma^2)^{-n/2}\exp\left\{-\frac{(y-X\beta)'(y-X\beta)}{2\sigma^2}\right\}.$$

**Student-t models M4–M6.** With $y_i\mid\beta,\sigma^2,\nu \sim t_\nu(x_i'\beta,\sigma^2)$ (location $x_i'\beta$, scale $\sigma$):
$$L(\beta,\sigma^2,\nu;y)=\prod_{i=1}^{n}\frac{\Gamma\!\left(\frac{\nu+1}{2}\right)}{\Gamma\!\left(\frac{\nu}{2}\right)\sqrt{\nu\pi\sigma^2}}\left[1+\frac{(y_i-x_i'\beta)^2}{\nu\sigma^2}\right]^{-\frac{\nu+1}{2}}.$$
There is no closed-form maximizer or conjugate prior for $\nu$, and the posterior of $(\beta,\sigma^2)$ is not standard.

**Augmented likelihood (M4–M6).** Use the scale mixture $y_i\mid\beta,\sigma^2,\omega_i\sim N(x_i'\beta,\sigma^2/\omega_i)$, $\omega_i\mid\nu\sim\text{Gamma}(\nu/2,\nu/2)$:
$$L(\beta,\sigma^2,\nu;y,\omega)=\prod_{i=1}^{n}\underbrace{\left(\frac{\omega_i}{2\pi\sigma^2}\right)^{1/2}\exp\left\{-\frac{\omega_i e_i^2}{2\sigma^2}\right\}}_{N(y_i;\,x_i'\beta,\ \sigma^2/\omega_i)}\ \underbrace{\frac{(\nu/2)^{\nu/2}}{\Gamma(\nu/2)}\,\omega_i^{\nu/2-1}e^{-\nu\omega_i/2}}_{\text{Gamma}(\omega_i;\,\nu/2,\,\nu/2)}$$
$$=(2\pi\sigma^2)^{-n/2}\left[\frac{(\nu/2)^{\nu/2}}{\Gamma(\nu/2)}\right]^{n}\prod_{i=1}^{n}\omega_i^{\frac{\nu+1}{2}-1}\exp\left\{-\sum_{i=1}^{n}\omega_i\left(\frac{e_i^2}{2\sigma^2}+\frac{\nu}{2}\right)\right\}.$$

**Check that integrating out $\omega_i$ gives back the t.** Write $u_i=e_i/\sigma$. The $\omega_i$ factor is a Gamma kernel, $\int_0^\infty \omega^{\frac{\nu+1}{2}-1}e^{-\omega(\nu+u_i^2)/2}d\omega=\Gamma\!\left(\tfrac{\nu+1}{2}\right)\left(\tfrac{\nu+u_i^2}{2}\right)^{-\frac{\nu+1}{2}}$. Then
$$\int p(y_i,\omega_i\mid\cdot)\,d\omega_i=\frac{(\nu/2)^{\nu/2}\,\Gamma\!\left(\frac{\nu+1}{2}\right)}{\sqrt{2\pi\sigma^2}\,\Gamma(\nu/2)}\left(\frac{\nu+u_i^2}{2}\right)^{-\frac{\nu+1}{2}}=\frac{\Gamma\!\left(\frac{\nu+1}{2}\right)}{\Gamma\!\left(\frac{\nu}{2}\right)\sqrt{\nu\pi\sigma^2}}\left(1+\frac{u_i^2}{\nu}\right)^{-\frac{\nu+1}{2}},$$
which is the $t_\nu(x_i'\beta,\sigma^2)$ density above. Given $\omega$, the model is a weighted Gaussian regression with weights $\omega_i$. That is what makes Gibbs sampling possible in 2(b).

#### Statement 2(b)

> **(b)** Obtenha amostras da distribuição a posteriori de cada um dos seis modelos via MCMC (por exemplo, Gibbs para M1–M3; Gibbs com os $\omega_i$ e um passo Metropolis–Hastings para $\nu$ em M4–M6; ou Stan/brms). Reporte média, desvio-padrão e intervalo de credibilidade de 95% de todos os parâmetros, e verifique a convergência das cadeias (traços, autocorrelações, tamanho efetivo).

### Answer 2(b): sampler

**Priors.** $\beta\sim N(m_0,V_0)$ with $m_0=(25,0,\dots)'$, $V_0=\operatorname{diag}(5^2,3^2,\dots)$; $\sigma^2\sim IG(a,b)$ with $a=3$, $b=50$; $\nu\sim\text{Gamma}(2,0.1)$.

**Full conditionals, Gaussian models.** The posterior kernel is
$$p(\beta,\sigma^2\mid y)\propto(\sigma^2)^{-n/2}e^{-\frac{(y-X\beta)'(y-X\beta)}{2\sigma^2}}\;e^{-\frac12(\beta-m_0)'V_0^{-1}(\beta-m_0)}\;(\sigma^2)^{-(a+1)}e^{-b/\sigma^2}.$$
- As a function of $\beta$, the exponent is $-\tfrac12\left[\beta'(V_0^{-1}+X'X/\sigma^2)\beta-2\beta'(V_0^{-1}m_0+X'y/\sigma^2)\right]$. Completing the square: $\beta\mid\sigma^2,y\sim N(m_1,V_1)$, $V_1=(V_0^{-1}+X'X/\sigma^2)^{-1}$, $m_1=V_1(V_0^{-1}m_0+X'y/\sigma^2)$.
- As a function of $\sigma^2$: $(\sigma^2)^{-(a+n/2)-1}\exp\{-[b+\tfrac12(y-X\beta)'(y-X\beta)]/\sigma^2\}$. So $\sigma^2\mid\beta,y\sim IG\!\left(a+\tfrac n2,\ b+\tfrac12\sum_i e_i^2\right)$.

The priors are independent rather than conjugate (the prior variance of $\beta$ does not scale with $\sigma^2$), so the joint posterior is not Normal-Inverse-Gamma. Gibbs sampling handles it directly.

**Full conditionals, Student-t models** (augmented likelihood of 2(a), $\Omega=\operatorname{diag}(\omega_i)$):
- $\beta\mid\sigma^2,\omega,y\sim N(m_1,V_1)$, $V_1=(V_0^{-1}+X'\Omega X/\sigma^2)^{-1}$, $m_1=V_1(V_0^{-1}m_0+X'\Omega y/\sigma^2)$: weighted least squares combined with the prior.
- $\sigma^2\mid\beta,\omega,y\sim IG\!\left(a+\tfrac n2,\ b+\tfrac12\sum_i\omega_ie_i^2\right)$.
- $\omega_i\mid\beta,\sigma^2,\nu,y\propto\omega_i^{\frac{\nu+1}{2}-1}e^{-\omega_i(\nu+e_i^2/\sigma^2)/2}$, that is $\omega_i\mid\cdot\sim\text{Gamma}\!\left(\tfrac{\nu+1}{2},\ \tfrac{\nu+e_i^2/\sigma^2}{2}\right)$. A large residual gives a small $\omega_i$, so that observation is down-weighted.
- $\nu$: $p(\nu\mid\beta,\sigma^2,y)\propto p(\nu)\prod_i t_\nu(e_i/\sigma)$ is not a standard distribution. We use a random-walk Metropolis–Hastings step on $\eta=\log\nu$: propose $\nu^*=\nu e^{\delta u}$ with $u\sim N(0,1)$ and $\delta=1.2$, and accept with probability
$$\min\left\{1,\ \frac{p(\nu^*)\prod_i t_{\nu^*}(e_i/\sigma)\ \nu^*}{p(\nu)\prod_i t_{\nu}(e_i/\sigma)\ \nu}\right\},$$
where the factor $\nu^*/\nu$ is the Jacobian of $\nu=e^\eta$. This step draws $\nu$ with $\omega$ integrated out, and the next step draws $\omega$ given the new $\nu$. Together they draw $(\nu,\omega)$ jointly from $p(\nu,\omega\mid\beta,\sigma^2,y)$, a valid blocked Gibbs step. It mixes much better than drawing $\nu\mid\omega$, because $\nu$ and the $\omega_i$ are strongly dependent.

**Implementation.** The sampler is vectorized over chains. Each chain carries a 0/1 mask saying which observations it uses. Here, 4 chains with all observations run in one array, starting from dispersed values drawn from the prior. In Question 3 the same code runs the 40 leave-one-out fits in one batch. Settings: 4 chains, 2,000 burn-in iterations and 10,000 kept per chain (40,000 draws per model), no thinning, `SEED = 2026`.

In [ ]:
MODELS = {  # name: (regressors, Student-t errors?)
    "M1": (["life"], False), "M2": (["ses"], False), "M3": (["life", "ses"], False),
    "M4": (["life"], True),  "M5": (["ses"], True),  "M6": (["life", "ses"], True),
}
ZCOL = {"life": z1, "ses": z2}
BNAME = {"life": "beta1", "ses": "beta2"}
A0, B0 = 3.0, 50.0        # sigma^2 ~ IG(A0, B0): shape, scale
NU_A, NU_B = 2.0, 0.1     # nu ~ Gamma(NU_A, rate NU_B)


def design(model):
    """Design matrix [1, z_j, ...] and coefficient names for model `model` ('M1'..'M6')."""
    regs, _ = MODELS[model]
    X = np.column_stack([np.ones(n)] + [ZCOL[r] for r in regs])
    return X, ["beta0"] + [BNAME[r] for r in regs]


def beta_prior(k, scale=1.0):
    """Prior means and s.d. of (beta0, beta_j...): N(25, 5^2) and N(0, 3^2), s.d. times `scale`."""
    m0 = np.r_[25.0, np.zeros(k - 1)]
    s0 = np.r_[5.0, np.full(k - 1, 3.0)] * scale
    return m0, s0


def log_t(u, nu):
    """Log density of the standard Student-t with nu d.f. at u (broadcasts)."""
    return (gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(nu * np.pi)
            - (nu + 1) / 2 * np.log1p(u**2 / nu))

In [ ]:
def gibbs(model, mask, n_keep=5000, burn=2000, prior_scale=1.0, seed=1, nu_step=1.2):
    """Batched Gibbs sampler (with one MH step for nu) for model M1..M6.

    mask : (B, n) array of 0/1. Row b says which observations enter chain b. Four rows of
           ones = four independent chains on the full data; row i with a 0 in column i =
           the fit without observation i (exact LOO), so 40 refits run as one batch.
    Returns dict with draws beta (S,B,k), sigma2 (S,B), nu (S,B) or None, omega posterior
    mean (B,n), nu acceptance rate (B,).
    """
    rng = np.random.default_rng(seed)
    X, names = design(model)
    student = MODELS[model][1]
    B, k = mask.shape[0], X.shape[1]
    m0, s0 = beta_prior(k, prior_scale)
    P0, P0m0 = np.diag(1 / s0**2), m0 / s0**2
    n_eff = mask.sum(1)
    # dispersed starting values (drawn from the prior)
    s2 = 1 / rng.gamma(A0, 1 / B0, B)
    nu = rng.gamma(NU_A, 1 / NU_B, B) if student else None
    omega = np.ones((B, n))
    S = n_keep
    out_b, out_s2 = np.empty((S, B, k)), np.empty((S, B))
    out_nu = np.empty((S, B)) if student else None
    om_sum, acc = np.zeros((B, n)), np.zeros(B)
    for it in range(burn + n_keep):
        lam = mask * omega                                   # precision weights
        prec = np.einsum("bi,ij,il->bjl", lam, X, X) / s2[:, None, None] + P0
        rhs = (lam * y) @ X / s2[:, None] + P0m0
        L = np.linalg.cholesky(prec)
        mean = np.linalg.solve(prec, rhs[..., None])[..., 0]
        zz = rng.standard_normal((B, k, 1))
        beta = mean + np.linalg.solve(np.swapaxes(L, 1, 2), zz)[..., 0]   # N(mean, prec^-1)
        e = y - beta @ X.T
        s2 = 1 / rng.gamma(A0 + n_eff / 2, 1 / (B0 + 0.5 * (lam * e**2).sum(1)))
        if student:
            u = e / np.sqrt(s2)[:, None]
            # target: log p(nu | beta, sigma2, y) + log-Jacobian, omega integrated out
            lpost = lambda v: ((mask * log_t(u, v[:, None])).sum(1)
                               + NU_A * np.log(v) - NU_B * v)
            prop = nu * np.exp(nu_step * rng.standard_normal(B))
            ok = np.log(rng.uniform(size=B)) < lpost(prop) - lpost(nu)
            nu = np.where(ok, prop, nu)
            omega = rng.gamma((nu[:, None] + 1) / 2, 2 / (nu[:, None] + u**2))
        if it >= burn:
            g = it - burn
            out_b[g], out_s2[g] = beta, s2
            if student:
                out_nu[g] = nu
                om_sum += omega
                acc += ok
    return {"model": model, "names": names, "beta": out_b, "sigma2": out_s2, "nu": out_nu,
            "omega_mean": om_sum / S if student else None,
            "acc_nu": acc / S if student else None}

In [ ]:
def ess(x):
    """Multi-chain effective sample size (Geyer initial monotone sequence). x: (S, C)."""
    S, C = x.shape
    xc = x - x.mean(0)
    f = np.fft.rfft(xc, n=2 * S, axis=0)
    acov = np.fft.irfft(f * np.conj(f), axis=0)[:S] / S
    W = (acov[0] * S / (S - 1)).mean()
    var_plus = W * (S - 1) / S + (x.mean(0).var(ddof=1) if C > 1 else 0.0)
    rho = 1 - (W - acov.mean(1)) / var_plus
    rho[0] = 1.0
    P = rho[: S - 1 : 2] + rho[1:S:2]
    neg = np.where(P <= 0)[0]
    P = P[: neg[0]] if len(neg) else P
    P = np.minimum.accumulate(P)
    tau = -1 + 2 * P.sum()
    return S * C / max(tau, 1e-12)


def rhat(x):
    """Split R-hat (Gelman et al.). x: (S, C)."""
    h = x.shape[0] // 2
    xs = np.concatenate([x[:h], x[h : 2 * h]], axis=1)
    W = xs.var(0, ddof=1).mean()
    Bv = h * xs.mean(0).var(ddof=1)
    return np.sqrt(((h - 1) / h * W + Bv / h) / W)


def draws_table(fit):
    """Dict param -> (S, C) array of draws (beta's, sigma2, sigma, nu)."""
    d = {nm: fit["beta"][:, :, j] for j, nm in enumerate(fit["names"])}
    d["sigma2"] = fit["sigma2"]
    d["sigma"] = np.sqrt(fit["sigma2"])
    if fit["nu"] is not None:
        d["nu"] = fit["nu"]
    return d


def summarize(fit):
    """Posterior mean, s.d., 95% credible interval, ESS and split R-hat per parameter."""
    rows = {}
    for nm, x in draws_table(fit).items():
        q = np.quantile(x, [0.025, 0.975])
        rows[nm] = {"mean": x.mean(), "sd": x.std(ddof=1), "2.5%": q[0], "97.5%": q[1],
                    "ESS": ess(x), "Rhat": rhat(x)}
    return pd.DataFrame(rows).T

In [ ]:
N_CHAINS, N_KEEP, N_BURN = 4, 10_000, 2_000
full = {}
for k, m in enumerate(MODELS):
    full[m] = gibbs(m, np.ones((N_CHAINS, n)), n_keep=N_KEEP, burn=N_BURN, seed=SEED + 10 * k)
print(f"{N_CHAINS} chains x ({N_BURN} burn-in + {N_KEEP} kept) iterations per model, "
      f"{N_CHAINS * N_KEEP} draws kept; elapsed {time.time() - T0:.0f} s")
for m in ["M4", "M5", "M6"]:
    print(f"{m}: MH acceptance rate for nu by chain = {np.round(full[m]['acc_nu'], 3)}")

In [ ]:
post = pd.concat({m: summarize(full[m]) for m in MODELS}, names=["model", "parameter"])
post.round({"mean": 3, "sd": 3, "2.5%": 3, "97.5%": 3, "ESS": 0, "Rhat": 4})

In [ ]:
def plot_chains(fit, lags=30, thin=25):
    """Trace plots (all chains, thinned for display) and autocorrelations (chain 1)."""
    d = draws_table(fit); d.pop("sigma")
    fig, ax = plt.subplots(len(d), 2, figsize=(11, 1.6 * len(d)), gridspec_kw={"width_ratios": [3, 1]})
    for r_, (nm, x) in enumerate(d.items()):
        for c in range(x.shape[1]):
            ax[r_, 0].plot(np.arange(0, len(x), thin), x[::thin, c], lw=0.4, alpha=0.7)
        ax[r_, 0].set_ylabel(nm)
        xc = x[:, 0] - x[:, 0].mean()
        ac = [1.0] + [np.dot(xc[:-l], xc[l:]) / np.dot(xc, xc) for l in range(1, lags + 1)]
        ax[r_, 1].bar(range(lags + 1), ac, width=0.7)
        ax[r_, 1].set_ylim(-0.1, 1)
    ax[0, 0].set_title(f"{fit['model']}: traces, {x.shape[1]} chains (every {thin}th draw)")
    ax[0, 1].set_title("autocorrelation, chain 1")
    ax[-1, 0].set_xlabel("iteration after burn-in"); ax[-1, 1].set_xlabel("lag")
    fig.tight_layout(); plt.show()

In [ ]:
plot_chains(full["M1"])

In [ ]:
plot_chains(full["M2"])

In [ ]:
plot_chains(full["M3"])

In [ ]:
plot_chains(full["M4"])

In [ ]:
plot_chains(full["M5"])

In [ ]:
plot_chains(full["M6"])

### Answer 2(b): posterior summaries and convergence

**Result (posterior mean, 95% credible interval):**

| | $\beta_0$ | $\beta_1$ (life) | $\beta_2$ (ses) | $\sigma$ | $\nu$ |
|---|---|---|---|---|---|
| M1 | 27.25 [25.67, 28.81] | 1.89 [0.33, 3.45] | – | 5.08 [4.14, 6.30] | – |
| M2 | 27.24 [25.70, 28.78] | – | −2.03 [−3.56, −0.50] | 5.03 [4.09, 6.24] | – |
| M3 | 27.25 [25.82, 28.67] | 2.19 [0.75, 3.61] | −2.31 [−3.71, −0.90] | 4.57 [3.71, 5.69] | – |
| M4 | 27.18 [25.60, 28.76] | 1.82 [0.24, 3.39] | – | 4.80 [3.76, 6.05] | 21.4 [4.6, 57.3] |
| M5 | 27.22 [25.70, 28.72] | – | −2.25 [−3.80, −0.66] | 4.52 [3.39, 5.81] | 16.9 [3.2, 49.4] |
| M6 | 27.27 [25.84, 28.68] | 2.01 [0.57, 3.46] | −2.39 [−3.84, −0.92] | 4.30 [3.34, 5.47] | 20.9 [4.3, 55.4] |

The table printed above also gives the posterior s.d. and $\sigma^2$. Interpretation:

- $\beta_0$ is about 27.2 to 27.3 in every model, close to $\bar y=27.30$, because the regressors are centred.
- One s.d. more life events (22.6 points) adds about 2 points of impairment. One s.d. more SES (25.3 points) removes about 2.3 points. Every 95% credible interval excludes zero: given the data and the prior, $\beta_1$ lies in $[0.75, 3.61]$ with probability 0.95 (M3).
- As predicted in 1(d), both coefficients are larger in absolute value when the other regressor is included (1.89 → 2.19 and −2.03 → −2.31), and $\sigma$ falls from 5.08/5.03 to 4.57.
- The posterior means sit slightly below the least-squares values in absolute value (2.34 and −2.46) because the $N(0,3^2)$ prior shrinks toward zero. The prior precision $1/9$ is about 6% of the posterior precision $1/0.724^2$.
- In M4–M6, $\sigma$ is a scale parameter, not the error s.d.: $\operatorname{Var}(\varepsilon_i)=\sigma^2\nu/(\nu-2)$. That is why it is smaller than in the Gaussian models.

**Convergence.**
- **Traces.** The four chains, started from dispersed prior draws, overlap from the first kept iteration, with no trend or sticking.
- **Autocorrelations.** The $\beta$'s are close to zero by lag 2 to 3. $\sigma^2$ is close to zero by lag 4 in the Gaussian models and by lag 8 in the t models, where it is coupled to $\nu$ through the $\omega_i$. $\nu$, updated by Metropolis–Hastings, decays by about lag 10.
- **Effective sizes.** ESS out of 40,000 draws ranges from 7,180 ($\nu$ in M5) to 40,551. The MH acceptance rate for $\nu$ is 0.51 to 0.53.
- **$\hat R$.** The maximum split $\hat R$ is 1.001.

No sign of non-convergence: every posterior summary above rests on at least 7,000 effective draws.

#### Statement 2(c)

> **(c)** Para os modelos M4–M6, compare graficamente a priori e a posteriori de $\nu$. Os dados trazem informação sobre $\nu$? Há evidência de caudas pesadas?

In [ ]:
grid = np.linspace(0.01, 100, 500)
prior_nu = stats.gamma(NU_A, scale=1 / NU_B)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for a, m in zip(ax, ["M4", "M5", "M6"]):
    a.hist(full[m]["nu"].ravel(), bins=np.arange(0, 101, 2.5), density=True, alpha=0.6,
           label="posterior (MCMC)")
    a.plot(grid, prior_nu.pdf(grid), "k", lw=2, label="prior Gamma(2, 0.1)")
    a.set_xlabel(r"$\nu$"); a.set_title(m); a.legend(fontsize=8)
ax[0].set_ylabel("density")
fig.suptitle(r"Figure. Prior and posterior of the degrees of freedom $\nu$")
fig.tight_layout(); plt.show()

nu_tab = {"prior": {"mean": prior_nu.mean(), "median": prior_nu.median(),
                    "2.5%": prior_nu.ppf(0.025), "97.5%": prior_nu.ppf(0.975),
                    "P(nu<10)": prior_nu.cdf(10), "P(nu>30)": prior_nu.sf(30)}}
for m in ["M4", "M5", "M6"]:
    v = full[m]["nu"].ravel()
    nu_tab[m] = {"mean": v.mean(), "median": np.median(v), "2.5%": np.quantile(v, 0.025),
                 "97.5%": np.quantile(v, 0.975), "P(nu<10)": (v < 10).mean(), "P(nu>30)": (v > 30).mean()}
pd.DataFrame(nu_tab).T.round(3)

In [ ]:
# Posterior means of the latent weights omega_i in M6: small omega_i = observation down-weighted
om = pd.Series(full["M6"]["omega_mean"].mean(0), index=np.arange(1, n + 1), name="E(omega_i | y)")
pd.concat([om, Mental.set_index(om.index)], axis=1).sort_values("E(omega_i | y)").head(6).round(3)

### Answer 2(c)

**Result:** the data carry little information about $\nu$. In M4 and M6 the posterior almost reproduces the prior, except that values below about 4 are ruled out. Only M5 shifts toward heavier tails, and only moderately. There is no evidence of heavy tails.

| | mean | median | 95% interval | $P(\nu<10)$ | $P(\nu>30)$ |
|---|---|---|---|---|---|
| prior Gamma(2, 0.1) | 20.0 | 16.8 | [2.4, 55.7] | 0.264 | 0.199 |
| M4 | 21.4 | 17.9 | [4.6, 57.3] | 0.199 | 0.214 |
| M5 | 16.9 | 13.5 | [3.2, 49.4] | 0.350 | 0.133 |
| M6 | 20.9 | 17.7 | [4.3, 55.4] | 0.216 | 0.208 |

- **What the data do say.** The only clear update in M4/M6 is in the lower tail: the 2.5% quantile rises from 2.4 to 4.3–4.6. Extremely heavy, Cauchy-like tails ($\nu$ below about 4) are incompatible with 40 residuals that contain no extreme value. Above that, the posterior follows the prior.
- **Heavy tails?** No. In M4 and M6 the posterior probability of $\nu<10$ (0.20 and 0.22) is below the prior's 0.26. M5 is the exception (0.35). Without `life`, some residuals become large and look like tail events. Once `life` enters (M6), the shift disappears, so it reflects a missing regressor, not the error distribution.
- **Latent weights.** The smallest posterior means of $\omega_i$ in M6 are 0.73 (observation 40), 0.82 (3), 0.85 (5 and 39). Their prior mean is 1. A real outlier under a heavy-tailed fit would get a weight well below 0.5, so no observation is strongly down-weighted.
- With $n=40$ the tail index is hard to learn: it is identified by the few largest residuals. This is why an informative prior on $\nu$ (Juárez & Steel 2010) is needed. Under a flat prior the posterior of $\nu$ would not be proper.

#### Statement 2(d)

> **(d)** Estime a verossimilhança marginal (preditiva a priori)
> $$p(y\mid M_k)=\int p(y\mid\theta_k,M_k)\,p(\theta_k\mid M_k)\,d\theta_k,$$
> para $k=1,\dots,6$, por pelo menos um método adequado (por exemplo, amostragem por importância com proposta $t$ multivariada centrada na moda a posteriori, bridge sampling (pacote `bridgesampling`), método de Chib, ou aproximação de Laplace). Evite o estimador da média harmônica e comente por quê. Dica: trabalhe sempre na escala logarítmica.

### Answer 2(d): method

$$p(y\mid M_k)=\int p(y\mid\theta_k,M_k)\,p(\theta_k\mid M_k)\,d\theta_k .$$

**Main method: importance sampling with a multivariate-t proposal at the posterior mode.**
1. **Unconstrained parametrization.** Work with $\phi=(\beta,\log\sigma^2[,\log\nu])$ so that a symmetric proposal fits. The integrand on the $\phi$ scale includes the Jacobian: $\tilde p(\phi)=p(y\mid\theta)\,p(\theta)\,\sigma^2[\,\nu\,]$. In M4–M6 the likelihood is the t likelihood of 2(a), with $\omega$ integrated out analytically, so the integral has dimension 3 to 5, not 43 to 45.
2. **Proposal.** Find the posterior mode $\hat\phi$ by BFGS and the Hessian $H$ of $-\log\tilde p$ at $\hat\phi$ by central differences. The proposal is $q=t_4(\hat\phi,H^{-1})$. Its tails are heavier than the posterior's, which keeps the weights bounded.
3. **Estimate.** Draw $\phi^{(m)}\sim q$, $m=1,\dots,M=100{,}000$. Then
$$\widehat{p}(y\mid M_k)=\frac1M\sum_m w_m,\qquad w_m=\frac{\tilde p(\phi^{(m)})}{q(\phi^{(m)})},$$
computed on the log scale as $\log\hat p=\operatorname{logsumexp}(\log w_m)-\log M$. Likelihoods of the order of $e^{-120}$ would otherwise underflow. The Monte Carlo s.e. of $\log\hat p$ is $\operatorname{sd}(w)/(\sqrt M\,\bar w)$.

**Cross-checks.**
- **Bridge sampling** (Meng & Wong 1996, iterative optimal bridge), with the same $q$ and 40,000 MCMC draws.
- **Laplace:** $\log\tilde p(\hat\phi)+\frac d2\log2\pi-\frac12\log|H|$.
- **Exact value for M1–M3.** Given $\sigma^2$, $\beta$ integrates out analytically: $y\mid\sigma^2\sim N(Xm_0,\ \sigma^2I+XV_0X')$. What remains is a one-dimensional integral over $\sigma^2$, done by quadrature.

**Why not the harmonic mean.** The identity $\dfrac{1}{p(y)}=E\!\left[\dfrac{1}{p(y\mid\theta)}\,\Big|\,y\right]$ is true. The estimator $\big[\frac1S\sum_s 1/p(y\mid\theta^{(s)})\big]^{-1}$ built from posterior draws is still a bad one (Newton & Raftery 1994; Neal 1994):
- $1/p(y\mid\theta)$ is largest where the likelihood is small. Those are regions where the prior has mass and the posterior almost never goes, so the expectation is dominated by draws that rarely appear.
- The variance of $1/p(y\mid\theta)$ under the posterior is typically infinite. The estimator converges extremely slowly, jumps when a rare draw arrives, and in typical runs is too high.
- It barely depends on the prior, while $p(y)$ depends on it strongly (see 2(g)). The estimator therefore misses the very thing that defines the marginal likelihood.

The output below shows all three problems.

In [ ]:
def log_joint(phi, model, prior_scale=1.0):
    """log p(y|theta) + log p(theta) + log Jacobian, on phi = (beta, log sigma2[, log nu]).

    phi : (M, d) array. Omega is integrated out analytically (Student-t likelihood).
    """
    phi = np.atleast_2d(phi)
    X, _ = design(model)
    student = MODELS[model][1]
    k = X.shape[1]
    beta, ls2 = phi[:, :k], phi[:, k]
    s2 = np.exp(ls2)
    e = y - beta @ X.T
    if student:
        lnu = phi[:, k + 1]
        nu = np.exp(lnu)
        ll = log_t(e / np.sqrt(s2)[:, None], nu[:, None]).sum(1) - n / 2 * ls2
        lp = stats.gamma.logpdf(nu, NU_A, scale=1 / NU_B) + lnu
    else:
        ll = -n / 2 * np.log(2 * np.pi * s2) - (e**2).sum(1) / (2 * s2)
        lp = 0.0
    m0, s0 = beta_prior(k, prior_scale)
    lp = lp + stats.norm.logpdf(beta, m0, s0).sum(1)
    lp = lp + stats.invgamma.logpdf(s2, A0, scale=B0) + ls2
    return ll + lp


def to_phi(fit):
    """Posterior draws on the unconstrained scale, pooled over chains: (S*C, d)."""
    cols = [fit["beta"].reshape(-1, fit["beta"].shape[2]), np.log(fit["sigma2"]).reshape(-1, 1)]
    if fit["nu"] is not None:
        cols.append(np.log(fit["nu"]).reshape(-1, 1))
    return np.hstack(cols)


def num_hessian(f, x, h=1e-4):
    """Central-difference Hessian of a scalar function f at x."""
    d = len(x)
    H = np.empty((d, d))
    for i in range(d):
        for j in range(i, d):
            ei, ej = np.eye(d)[i] * h, np.eye(d)[j] * h
            H[i, j] = H[j, i] = (f(x + ei + ej) - f(x + ei - ej) - f(x - ei + ej)
                                 + f(x - ei - ej)) / (4 * h * h)
    return H


def mode_and_cov(fit, prior_scale=1.0):
    """Posterior mode on phi and the inverse negative Hessian there."""
    f = lambda p: -log_joint(p, fit["model"], prior_scale)[0]
    res = optimize.minimize(f, to_phi(fit).mean(0), method="BFGS")
    H = num_hessian(f, res.x)
    return res.x, np.linalg.inv(H), -res.fun, H


def logml_is(fit, prior_scale=1.0, M=100_000, df=4, seed=11):
    """Importance-sampling estimate of log p(y|M) with a multivariate-t proposal at the mode.

    Returns (estimate, Monte Carlo s.e. of the estimate, proposal object)."""
    mode, Sig, _, _ = mode_and_cov(fit, prior_scale)
    q = stats.multivariate_t(loc=mode, shape=Sig, df=df, seed=seed)
    th = q.rvs(M)
    lw = log_joint(th, fit["model"], prior_scale) - q.logpdf(th)
    est = logsumexp(lw) - np.log(M)
    w = np.exp(lw - lw.max())
    se = w.std(ddof=1) / np.sqrt(M) / w.mean()
    return est, se, q


def logml_bridge(fit, q, prior_scale=1.0, M=50_000, n_iter=200):
    """Bridge sampling (Meng & Wong 1996 optimal bridge), proposal q, posterior draws from fit."""
    post = to_phi(fit)
    post = post[np.random.default_rng(5).choice(len(post), min(M, len(post)), replace=False)]
    prop = q.rvs(M)
    l1 = log_joint(post, fit["model"], prior_scale) - q.logpdf(post)   # posterior draws
    l2 = log_joint(prop, fit["model"], prior_scale) - q.logpdf(prop)   # proposal draws
    N1, N2 = len(l1), len(l2)
    s1, s2 = N1 / (N1 + N2), N2 / (N1 + N2)
    lr = 0.0
    for _ in range(n_iter):
        num = logsumexp(l2 - np.logaddexp(np.log(s1) + l2, np.log(s2) + lr)) - np.log(N2)
        den = logsumexp(-np.logaddexp(np.log(s1) + l1, np.log(s2) + lr)) - np.log(N1)
        new = num - den
        if abs(new - lr) < 1e-10:
            break
        lr = new
    return lr


def logml_laplace(fit, prior_scale=1.0):
    """Laplace approximation of log p(y|M) on the unconstrained scale."""
    mode, Sig, lmax, H = mode_and_cov(fit, prior_scale)
    d = len(mode)
    return lmax + d / 2 * np.log(2 * np.pi) - 0.5 * np.linalg.slogdet(H)[1]


def logml_exact_gauss(model, prior_scale=1.0):
    """Exact log p(y|M) for M1-M3: beta integrated out analytically, then 1-D quadrature."""
    X, _ = design(model)
    m0, s0 = beta_prior(X.shape[1], prior_scale)
    lam, U = np.linalg.eigh(X @ np.diag(s0**2) @ X.T)
    r2 = (U.T @ (y - X @ m0)) ** 2
    def f(ls2):
        s2 = np.exp(ls2)
        return (-n / 2 * np.log(2 * np.pi) - 0.5 * np.log(s2 + lam).sum()
                - 0.5 * (r2 / (s2 + lam)).sum()
                + stats.invgamma.logpdf(s2, A0, scale=B0) + ls2)
    c = -optimize.minimize_scalar(lambda t: -f(t), bounds=(-5, 10), method="bounded").fun
    val, _ = integrate.quad(lambda t: np.exp(f(t) - c), -10, 12, limit=200)
    return c + np.log(val)


def logml_harmonic(fit):
    """Harmonic-mean estimator (shown only to illustrate why it is avoided)."""
    phi = to_phi(fit)
    X, _ = design(fit["model"])
    k = X.shape[1]
    ll = log_joint(phi, fit["model"])  # includes prior; remove it below
    m0, s0 = beta_prior(k)
    lp = stats.norm.logpdf(phi[:, :k], m0, s0).sum(1) + stats.invgamma.logpdf(
        np.exp(phi[:, k]), A0, scale=B0) + phi[:, k]
    if fit["nu"] is not None:
        lp = lp + stats.gamma.logpdf(np.exp(phi[:, k + 1]), NU_A, scale=1 / NU_B) + phi[:, k + 1]
    ll = ll - lp
    return -(logsumexp(-ll) - np.log(len(ll)))

In [ ]:
ml = {}
for m in MODELS:
    est, se, q = logml_is(full[m])
    ml[m] = {"IS (t proposal)": est, "IS MC s.e.": se,
             "bridge": logml_bridge(full[m], q), "Laplace": logml_laplace(full[m]),
             "exact (M1-M3)": logml_exact_gauss(m) if not MODELS[m][1] else np.nan,
             "harmonic mean": logml_harmonic(full[m])}
ml = pd.DataFrame(ml).T
display(ml.round(3))

# Harmonic mean, chain by chain: the estimator is unstable and biased upward
hm_chain = {m: [logml_harmonic({**full[m], "beta": full[m]["beta"][:, [c]],
                                "sigma2": full[m]["sigma2"][:, [c]],
                                "nu": None if full[m]["nu"] is None else full[m]["nu"][:, [c]]})
                for c in range(N_CHAINS)] for m in MODELS}
pd.DataFrame(hm_chain, index=[f"chain {c + 1}" for c in range(N_CHAINS)]).T.round(3)

### Answer 2(d)

**Result (importance sampling):** $\log p(y\mid M_k) = -125.927$ (M1), $-125.481$ (M2), $-122.517$ (M3), $-125.945$ (M4), $-124.743$ (M5), $-122.481$ (M6), each with Monte Carlo s.e. ≤ 0.002.

- **Accuracy.** For M1–M3, importance sampling agrees with the exact value to within 0.002. Bridge sampling agrees with importance sampling to within 0.006 in all six models. Laplace is off by at most 0.09: the posterior is close to Gaussian on the $(\beta,\log\sigma^2,\log\nu)$ scale, which is also why the t proposal works well.
- **Harmonic mean, the estimator to avoid.**
  - It is too high by 1.3 to 2.4 nats (M3: −120.081 against the exact −122.517).
  - It differs by 0.31 to 0.75 nats between chains of the same run.
  - Its error is larger for the bigger models. It would report $2\log B_{31} = 2(-120.081+124.338)=8.51$ (strong evidence) where the correct value is 6.82. Ranking models with it would overstate the evidence for the larger models.

#### Statement 2(e)

> **(e)** Monte uma tabela com $\log p(y\mid M_k)$, os log-fatores de Bayes $\log B_{k*}=\log p(y\mid M_k)-\log p(y\mid M_*)$ em relação ao melhor modelo $M_*$, e as probabilidades a posteriori dos modelos supondo $P(M_k)=1/6$. Interprete usando a escala de Kass & Raftery (1995).

In [ ]:
def kass_raftery(two_log_b):
    """Kass & Raftery (1995) category for 2 log B (evidence for the better model)."""
    v = abs(two_log_b)
    return ("not worth more than a bare mention" if v < 2 else "positive" if v < 6
            else "strong" if v < 10 else "very strong")


def bf_table(logml):
    """log p(y|M_k), log B_k* against the best model, 2 log B, P(M_k|y) with equal prior odds."""
    t_ = pd.DataFrame({"log p(y|M)": logml})
    t_["log B_k*"] = t_["log p(y|M)"] - t_["log p(y|M)"].max()
    t_["2 log B_*k"] = 2 * (t_["log p(y|M)"].max() - t_["log p(y|M)"])
    t_["P(M_k|y)"] = np.exp(t_["log B_k*"]) / np.exp(t_["log B_k*"]).sum()
    t_["K&R (vs best)"] = [("best model" if v == 0 else kass_raftery(v)) for v in t_["2 log B_*k"]]
    return t_

LOGML = ml.iloc[:, 0]          # importance-sampling estimates
bf = bf_table(LOGML)
bf.round(3)

### Answer 2(e)

**Result:** M6 has the largest marginal likelihood, but M3 is tied with it ($\log B_{3*}=-0.036$). The two-regressor models together hold 90% of the posterior probability.

| model | $\log p(y\mid M_k)$ | $\log B_{k*}$ | $2\log B_{*k}$ | $P(M_k\mid y)$ | Kass & Raftery (evidence against $M_k$) |
|---|---|---|---|---|---|
| M6 | −122.481 | 0 | 0 | 0.458 | best model |
| M3 | −122.517 | −0.036 | 0.07 | 0.442 | not worth more than a bare mention |
| M5 | −124.743 | −2.262 | 4.52 | 0.048 | positive |
| M2 | −125.481 | −3.000 | 6.00 | 0.023 | strong (on the boundary with positive) |
| M1 | −125.927 | −3.446 | 6.89 | 0.015 | strong |
| M4 | −125.945 | −3.463 | 6.93 | 0.014 | strong |

$P(M_k\mid y)=B_{k*}/\sum_j B_{j*}$ with equal prior probabilities 1/6. The Kass & Raftery (1995) scale for $2\log B$ is: 0–2 not worth more than a bare mention, 2–6 positive, 6–10 strong, >10 very strong.

- **M6 and M3.** The Monte Carlo s.e. of each $\log p(y\mid M_k)$ is ≤ 0.002, so the order M6 > M3 is numerically stable. As evidence, though, a Bayes factor of $e^{0.036}=1.04$ is nothing.
- **Single-regressor models.** The best one, M5, is $e^{2.26}\approx 10$ times less supported than M6 (positive evidence). M2, M1 and M4 are 20 to 32 times less supported (strong evidence).

#### Statement 2(f)

> **(f)** Responda: (i) as duas variáveis explicativas são necessárias? (ii) erros Student-t são preferidos aos Gaussianos?

In [ ]:
pairs_bf = [("M3", "M1"), ("M3", "M2"), ("M6", "M4"), ("M6", "M5"),   # (i) both regressors?
            ("M4", "M1"), ("M5", "M2"), ("M6", "M3")]                 # (ii) Student-t vs Gaussian?
pb = pd.DataFrame({f"{a} vs {b}": {"log B": LOGML[a] - LOGML[b], "B": np.exp(LOGML[a] - LOGML[b]),
                                   "2 log B": 2 * (LOGML[a] - LOGML[b]),
                                   "K&R": kass_raftery(2 * (LOGML[a] - LOGML[b]))}
                   for a, b in pairs_bf}).T
display(pb)
P = bf["P(M_k|y)"]
print(f"P(both regressors | y) = P(M3) + P(M6) = {P['M3'] + P['M6']:.3f}")
print(f"P(Student-t errors | y) = P(M4) + P(M5) + P(M6) = {P['M4'] + P['M5'] + P['M6']:.3f}")

### Answer 2(f)

**(i) Are both regressors needed? Yes.** Each two-regressor model beats both of its one-regressor counterparts:

| comparison | $B$ | $2\log B$ | Kass & Raftery |
|---|---|---|---|
| M3 vs M1 (add ses) | 30.3 | 6.82 | strong |
| M3 vs M2 (add life) | 19.4 | 5.93 | positive (near strong) |
| M6 vs M4 (add ses) | 31.9 | 6.93 | strong |
| M6 vs M5 (add life) | 9.6 | 4.52 | positive |

$P(\text{both regressors}\mid y)=P(M_3\mid y)+P(M_6\mid y)=0.901$. Adding `ses` is supported more strongly than adding `life`, in line with $|r(y,x_2)|>|r(y,x_1)|$ in 1(c).

**(ii) Are Student-t errors preferred? No, and not rejected either.** With the same mean function, the Bayes factors are M6 vs M3 1.04, M4 vs M1 0.98, M5 vs M2 2.09. All are "not worth more than a bare mention" ($|2\log B|<2$). $P(\text{t errors}\mid y)=0.520$, against a prior probability of 0.5. The data cannot tell the two error distributions apart, which matches 2(c): the posterior of $\nu$ is almost its prior. With equal evidence, the Gaussian model M3 is the natural choice because it is simpler.

#### Statement 2(g)

> **(g)** Sensibilidade à priori. Repita o item (e) multiplicando por 3 os desvios-padrão a priori de $\beta_0$, $\beta_1$ e $\beta_2$. Os fatores de Bayes mudam? E as posterioris dos coeficientes? Relacione com o paradoxo de Lindley–Bartlett.

In [ ]:
SCALE = 3.0      # prior s.d. of beta0, beta1, beta2 multiplied by 3; sigma2 and nu priors unchanged
full_d, ml_d = {}, {}
for k, m in enumerate(MODELS):
    full_d[m] = gibbs(m, np.ones((N_CHAINS, n)), n_keep=N_KEEP, burn=N_BURN,
                      prior_scale=SCALE, seed=SEED + 300 + 10 * k)
    est, se, _ = logml_is(full_d[m], prior_scale=SCALE)
    ml_d[m] = {"IS": est, "IS MC s.e.": se,
               "exact (M1-M3)": logml_exact_gauss(m, SCALE) if not MODELS[m][1] else np.nan}
ml_d = pd.DataFrame(ml_d).T
bf_d = bf_table(ml_d["IS"])
comp = pd.DataFrame({"log p(y|M), base": LOGML, "log p(y|M), diffuse": bf_d["log p(y|M)"],
                     "change": bf_d["log p(y|M)"] - LOGML,
                     "log B_k*, base": bf["log B_k*"], "log B_k*, diffuse": bf_d["log B_k*"],
                     "P(M|y), base": bf["P(M_k|y)"], "P(M|y), diffuse": bf_d["P(M_k|y)"]})
display(ml_d.round(3))
comp.round(3)

In [ ]:
pd_ = pd.concat({"base prior": post[["mean", "sd", "2.5%", "97.5%"]],
                 "diffuse prior (s.d. x3)": pd.concat({m: summarize(full_d[m]) for m in MODELS},
                                                      names=["model", "parameter"])[["mean", "sd", "2.5%", "97.5%"]]},
                axis=1)
pd_.loc[pd_.index.get_level_values(1).str.startswith("beta")].round(3)

In [ ]:
LOGML_D = bf_d["log p(y|M)"]
Pd = bf_d["P(M_k|y)"]
print(f"P(both regressors | y): base {P['M3'] + P['M6']:.3f}, diffuse {Pd['M3'] + Pd['M6']:.3f}")
pd.DataFrame({f"{a} vs {b}": {"2 log B, base": 2 * (LOGML[a] - LOGML[b]),
                              "2 log B, diffuse": 2 * (LOGML_D[a] - LOGML_D[b])}
              for a, b in pairs_bf}).T.round(3)

### Answer 2(g)

**Result:**
- **Bayes factors.** All marginal likelihoods fall, and the Bayes factors for the larger models shrink. Evidence for adding the second regressor drops from strong to positive: $2\log B_{31}$ goes from 6.82 to 5.34 and $2\log B_{64}$ from 6.93 to 5.45. $P(M_3\cup M_6\mid y)$ goes from 0.901 to 0.804.
- **Coefficient posteriors.** They barely change.

**Marginal likelihoods** (importance sampling; the exact values for M1–M3 agree to within 0.002):

| | base | diffuse (s.d. ×3) | change |
|---|---|---|---|
| M1 | −125.927 | −127.802 | −1.874 |
| M2 | −125.481 | −127.329 | −1.847 |
| M3 | −122.517 | −125.130 | −2.613 |
| M4 | −125.945 | −127.837 | −1.892 |
| M5 | −124.743 | −126.540 | −1.796 |
| M6 | −122.481 | −125.109 | −2.628 |

**Bayes factors.** $2\log B$, base → diffuse:
- M3 vs M1: 6.82 → 5.34 (strong → positive)
- M3 vs M2: 5.93 → 4.40
- M6 vs M4: 6.93 → 5.45 (strong → positive)
- M6 vs M5: 4.52 → 2.86

The t vs Gaussian comparisons do not move (M6 vs M3: 0.07 → 0.04), because those models have the same number of coefficients. M6 and M3 remain tied at the top, with probabilities 0.406 and 0.398.

**Coefficient posteriors.** In M3, $\beta_1$ goes from 2.19 [0.75, 3.61] to 2.32 [0.87, 3.78], and $\beta_2$ from −2.31 to −2.44. The posterior s.d. grow by 2–3% (0.724 → 0.740). The means move toward least squares (2.34, −2.46) because the prior shrinks less. Conclusions about the coefficients are robust; the Bayes factors are not.

**Why.** By the Laplace approximation, $\log p(y\mid M)\approx\log p(y\mid\hat\theta)+\log p(\hat\theta)+\frac d2\log 2\pi+\frac12\log|\Sigma_{post}|$. Widening the priors hardly changes the likelihood term or the posterior covariance, but it lowers the prior density at $\hat\theta$. For a coefficient whose prior s.d. goes from $s$ to $3s$, the change is
$$\Delta\log p(\hat\theta_j)=-\log 3+\frac{(\hat\theta_j-m_j)^2}{2s^2}\left(1-\tfrac19\right).$$
That is −1.00 for $\beta_0$ ($\hat\beta_0\approx27.3$, $m=25$, $s=5$) and about −0.86 and −0.84 for $\beta_1$ and $\beta_2$ ($|\hat\beta|\approx2.2$–$2.3$, $s=3$). This predicts about −1.86 for M1/M2 (observed −1.85 to −1.87) and about −2.7 for M3 (observed −2.61). The intercept's cost is common to all models and cancels in every Bayes factor. What remains is 0.74 to 0.83 nat charged to each extra coefficient.

**Lindley–Bartlett paradox.** For fixed data, as the prior variance of the parameter being tested grows, the Bayes factor for the larger model goes to zero. The prior spreads its mass over values the data rule out, and the marginal likelihood pays for that volume. A vague enough prior on $\beta_1$ would make the Bayes factor favour M2 over M3, even though the frequentist test of $\rho(y,x_1)=0$ rejects (p-value 0.018) and the posterior interval of $\beta_1$ excludes zero. Tripling the s.d. shows the start of this: the evidence weakens by one Kass–Raftery category while the posterior of $\beta_1$ is unchanged. Bayes factors therefore need proper priors on the tested parameters, elicited on a meaningful scale, as the standardized parametrization of this list allows. "Vague" is not neutral, and with improper priors the Bayes factor is not even defined.

---
## Question 3: Comparação de modelos via validação cruzada leave-one-out

#### Statement 3 (common part)

> Para cada observação $i=1,\dots,40$ e para cada modelo $M_k$, ajuste o modelo com as prioris do item 2 usando apenas os dados sem a observação $i$, $y_{(-i)}$, e obtenha a distribuição preditiva a posteriori de $y_i$:
> $$p(y_i\mid y_{(-i)},M_k)=\int p(y_i\mid\theta_k,M_k)\,p(\theta_k\mid y_{(-i)},M_k)\,d\theta_k\approx\frac1G\sum_{g=1}^{G}p\big(y_i\mid\theta_k^{(g)},M_k\big),$$
> em que $\theta_k^{(1)},\dots,\theta_k^{(G)}$ são amostras de $p(\theta_k\mid y_{(-i)},M_k)$. Isso requer $6\times40=240$ ajustes (pense em como reaproveitar código e paralelizar).

### Answer 3: method

**240 fits as 6 batched runs.** In the sampler of 2(b), chain $i$ uses the mask $1-e_i$: every observation except $i$. A run with 40 chains therefore produces the 40 exact leave-one-out posteriors $p(\theta\mid y_{(-i)},M_k)$ at once, one per chain, at roughly the cost of a single fit. For the t models, the $\omega_i$ of the left-out observation does not enter any full conditional. Settings per fit: 2,000 burn-in iterations and $G = 5{,}000$ kept draws, seeds `SEED + 500 + 10k`.

**Predictive quantities for $y_i$.** From the draws $\theta^{(g)}$ of the fit without $i$, with $\mu_i^{(g)}=x_i'\beta^{(g)}$:
- Log predictive density:
$$\log p(y_i\mid y_{(-i)})\approx\operatorname{logsumexp}_g\log p(y_i\mid\theta^{(g)})-\log G.$$
  Here $p(y_i\mid\theta)$ is $N(\mu_i,\sigma^2)$ in M1–M3. In M4–M6 it is the $t_\nu(\mu_i,\sigma^2)$ density, with $\omega_i$ integrated out: the left-out observation has no posterior for its own $\omega_i$.
- Predictive mean: $\hat y_{i}=\frac1G\sum_g\mu_i^{(g)}$. For the t models this needs $\nu>1$, and the posterior mass below that is negligible (the 2.5% quantile of $\nu$ is above 3).
- Predictive median: $\tilde y_i$ solves $\frac1G\sum_g F\!\left((\tilde y_i-\mu_i^{(g)})/\sigma^{(g)}\right)=\tfrac12$, the median of the mixture, found by Brent's method. $F$ is the $N(0,1)$ or the $t_{\nu^{(g)}}$ c.d.f.
- The Monte Carlo s.e. of each $\log p(y_i\mid y_{(-i)})$ comes from the delta method, and these are added in quadrature for the LPD.

In [ ]:
def loo_exact(fit):
    """From a batched LOO fit (chain i omits obs i): predictive mean, median, log density.

    Also returns the Monte Carlo s.e. of each log predictive density (delta method,
    treating the draws as independent)."""
    X, _ = design(fit["model"])
    student = fit["nu"] is not None
    pmean, pmed, lpd, lse = np.empty(n), np.empty(n), np.empty(n), np.empty(n)
    for i in range(n):
        mu = fit["beta"][:, i, :] @ X[i]
        sd = np.sqrt(fit["sigma2"][:, i])
        if student:
            nu = fit["nu"][:, i]
            lp = log_t((y[i] - mu) / sd, nu) - np.log(sd)
            cdf = lambda v: stats.t.cdf((v - mu) / sd, nu).mean() - 0.5
        else:
            lp = stats.norm.logpdf(y[i], mu, sd)
            cdf = lambda v: stats.norm.cdf((v - mu) / sd).mean() - 0.5
        lpd[i] = logsumexp(lp) - np.log(len(lp))
        p = np.exp(lp - lp.max())
        lse[i] = p.std(ddof=1) / np.sqrt(len(p)) / p.mean()
        pmean[i] = mu.mean()
        pmed[i] = optimize.brentq(cdf, mu.min() - 50, mu.max() + 50, xtol=1e-6)
    return pd.DataFrame({"y": y, "pred_mean": pmean, "pred_median": pmed, "lpd": lpd, "lpd_se": lse})

In [ ]:
LOO_KEEP, LOO_BURN = 5_000, 2_000
def run_loo(prior_scale=1.0, seed0=SEED + 500):
    """Exact LOO for all six models: one batched run of 40 chains per model (chain i omits obs i)."""
    out = {}
    for k, m in enumerate(MODELS):
        f = gibbs(m, 1 - np.eye(n), n_keep=LOO_KEEP, burn=LOO_BURN, prior_scale=prior_scale,
                  seed=seed0 + 10 * k)
        out[m] = loo_exact(f)
    return out

loo = run_loo()
print(f"6 models x 40 leave-one-out fits, {LOO_BURN} burn-in + {LOO_KEEP} kept each; "
      f"elapsed {time.time() - T0:.0f} s")

#### Statement 3(a)

> **(a)** Para cada modelo, calcule os três critérios abaixo, avaliados na observação deixada de fora:
> - Erro quadrático médio: $\text{MSE}_k=\frac1n\sum_i(y_i-\hat y_{i,k})^2$, com $\hat y_{i,k}=E(y_i\mid y_{(-i)},M_k)$;
> - Erro absoluto médio: $\text{MAE}_k=\frac1n\sum_i|y_i-\tilde y_{i,k}|$, com $\tilde y_{i,k}$ a mediana de $p(y_i\mid y_{(-i)},M_k)$;
> - Escore logarítmico preditivo: $\text{LPD}_k=\sum_i\log p(y_i\mid y_{(-i)},M_k)$ (quanto maior, melhor).
>
> Por que faz sentido usar a média preditiva no MSE e a mediana preditiva no MAE?

#### Statement 3(b)

> **(b)** Apresente uma tabela com MSE, MAE e LPD dos seis modelos e o ordenamento dos modelos segundo cada critério. Os três critérios concordam entre si?

In [ ]:
def criteria(loo):
    """MSE (predictive mean), MAE (predictive median), LPD and its MC s.e., with ranks."""
    t_ = pd.DataFrame({m: {"MSE": ((d.y - d.pred_mean) ** 2).mean(),
                           "MAE": (d.y - d.pred_median).abs().mean(),
                           "LPD": d.lpd.sum(), "LPD MC s.e.": np.sqrt((d.lpd_se ** 2).sum())}
                       for m, d in loo.items()}).T
    t_["rank MSE"] = t_["MSE"].rank().astype(int)
    t_["rank MAE"] = t_["MAE"].rank().astype(int)
    t_["rank LPD"] = (-t_["LPD"]).rank().astype(int)
    return t_

crit = criteria(loo)
crit.round(3)

### Answer 3(a)

$\text{MSE}_k=\frac1n\sum_i(y_i-\hat y_{i,k})^2$, $\text{MAE}_k=\frac1n\sum_i|y_i-\tilde y_{i,k}|$ and $\text{LPD}_k=\sum_i\log p(y_i\mid y_{(-i)},M_k)$ are in the table above, computed from the leave-one-out predictive distributions.

**Why the mean for MSE and the median for MAE.** Each point forecast is the Bayes decision for the loss it is scored with. For any predictive distribution of $Y$:
- $E[(Y-a)^2]$ is minimized at $a=E(Y)$, since $\frac{d}{da}E[(Y-a)^2]=-2(E(Y)-a)=0$.
- $E|Y-a|$ is minimized at the median, since $\frac{d}{da}E|Y-a|=P(Y<a)-P(Y>a)=0$ gives $P(Y\le a)=\tfrac12$.

Scoring the mean with absolute error, or the median with squared error, would penalize a model for using the wrong summary of its own predictive distribution rather than for predicting badly. The LPD needs no point forecast: it scores the whole predictive density (a proper scoring rule).

### Answer 3(b)

**Result:** M6 is first and M3 second by all three criteria, with negligible gaps between them. The criteria disagree only on the order of the weaker one-regressor models.

| | MSE | MAE | LPD (MC s.e.) | rank MSE | rank MAE | rank LPD |
|---|---|---|---|---|---|---|
| M1 | 27.774 | 4.264 | −124.190 (0.026) | 6 | 6 | 4 |
| M2 | 27.656 | 3.946 | −124.321 (0.034) | 4 | 3 | 6 |
| M3 | 23.263 | 3.804 | −120.358 (0.034) | 2 | 2 | 2 |
| M4 | 27.772 | 4.264 | −124.288 (0.025) | 5 | 5 | 5 |
| M5 | 27.463 | 3.960 | −123.445 (0.029) | 3 | 4 | 3 |
| M6 | 23.167 | 3.792 | −120.290 (0.032) | 1 | 1 | 1 |

- **What the criteria agree on.** The two-regressor models predict clearly better: the LOO MSE is 23.2 against 27.5–27.8, a 16% reduction, and the LOO RMSE is about 4.8 against $s_y=5.46$. M6 is ahead of M3 by 0.10 in MSE, 0.012 in MAE and 0.068 in LPD. The LPD gap is only about 1.5 times its Monte Carlo s.e. ($\sqrt{0.034^2+0.032^2}=0.047$), so the order of M6 and M3 is not meaningful.
- **Where they disagree: the one-regressor models.** M2 is 4th by MSE, 3rd by MAE and last by LPD. M1 and M4 are tied on MSE (27.774 vs 27.772) and MAE. MSE and MAE judge only the centre of each predictive distribution, while LPD also judges its spread and tails, so a model can have slightly better point forecasts and a slightly worse predictive density. The differences among these four models are small (at most 0.31 in MSE and 0.88 in LPD).

#### Statement 3(c)

> **(c)** Faça um gráfico das contribuições individuais $\log p(y_i\mid y_{(-i)},M_6)-\log p(y_i\mid y_{(-i)},M_3)$ contra $i$ (ou contra $y_i$). Quais observações mais favorecem os erros Student-t? Elas aparecem como atípicas nos gráficos do item 1?

In [ ]:
idx = np.arange(1, n + 1)
d63 = pd.Series((loo["M6"].lpd - loo["M3"].lpd).values, index=idx)
top = d63.sort_values(ascending=False).index[:5]     # observation numbers (1-based)
low = d63.sort_values().index[:5]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].vlines(idx, 0, d63.values, lw=2); ax[0].axhline(0, color="k", lw=0.8)
ax[0].set_xlabel("observation i (file order)")
ax[1].scatter(y, d63.values, s=18); ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_xlabel(r"$y_i$")
for a, xv in zip(ax, [idx, y]):
    a.set_ylabel(r"$\log p(y_i|y_{(-i)},M_6) - \log p(y_i|y_{(-i)},M_3)$")
    for i in list(top) + list(low):
        a.annotate(str(i), (xv[i - 1], d63[i]), fontsize=8, xytext=(3, 2), textcoords="offset points")
fig.suptitle("Figure. Pointwise LOO log-score difference, Student-t (M6) minus Gaussian (M3)")
fig.tight_layout(); plt.show()

# LOO residual: y_i minus the M3 predictive mean computed without observation i
loo_res = (loo["M3"].y - loo["M3"].pred_mean).values
tab = pd.DataFrame({"y": y, "life": life, "ses": ses, "LOO residual (M3)": loo_res,
                    "LOO residual / sigma": loo_res / post.loc[("M3", "sigma"), "mean"],
                    "diff M6-M3": d63.values}, index=idx)
print(f"sum of differences = LPD(M6) - LPD(M3) = {d63.sum():.3f}; "
      f"{(d63 > 0).sum()} observations favour M6, {(d63 < 0).sum()} favour M3")
print(f"obs. 22 and 31 (life = 97 and 95, beyond the fence in 1(a)): diff = {d63[22]:.3f} and {d63[31]:.3f}")
pd.concat({"most favour M6": tab.loc[top], "most favour M3": tab.loc[low]}).round(3)

### Answer 3(c)

**Result:** observation 40 favours the Student-t errors most (+0.198; $y=41$, life = 89, ses = 75), followed by observation 39 (+0.087; $y=41$, life = 49, ses = 3). The total advantage of M6 is only $\text{LPD}_6-\text{LPD}_3=0.068$, with 26 observations favouring M6 and 14 favouring M3.

- **Who gains.** The two observations that favour M6 most have large positive LOO residuals under M3: +12.7 (2.78$\sigma$) for observation 40 and +9.7 (2.12$\sigma$) for 39. The t predictive has heavier tails and a smaller scale ($\sigma$ = 4.30 vs 4.57), so it is more peaked. The many observations near their prediction gain a little (18, 27, 13: +0.04 to +0.06).
- **Who loses.** The largest losses are observations 5 (−0.22) and 3 (−0.13), both with $y=20$ and LOO residuals near −2$\sigma$, then 36, 6 and 35 (residuals of 1.0 to 1.4$\sigma$). Observation 39 (+2.1$\sigma$) gains while 3 (−2.0$\sigma$) loses, so the sign is not a simple function of the residual size. Apart from observation 40, the t model has no systematic advantage.
- **Are they atypical in item 1?** Partly.
  - Observations 39 and 40 are the two largest values of $y$ (41), separated from the rest of the histogram by an empty bin, but inside the Tukey fence (41.9).
  - In the scatter plots, observation 40 is the clearest point off the $y\times x_2$ line: high impairment with high SES. Observation 39 sits on the SES trend (it has the lowest SES), which is why its residual is smaller than that of 40.
  - The observations flagged in 1(a), 22 and 31 (life = 97 and 95), gain little (+0.040 and +0.023). They are extreme in $x$ (leverage), not in $y$ given $x$ (residual), and only the residual matters for the error distribution.

#### Statement 3(d)

> **(d)** Compare as conclusões da validação cruzada com as do fator de Bayes (item 2). Lembre que $\exp(\text{LPD}_k)$ é o produto das ordenadas preditivas condicionais (CPO) e que $\exp(\text{LPD}_k-\text{LPD}_l)$ é o pseudo-fator de Bayes de Geisser & Eddy (1979). Por que os dois enfoques podem discordar? Qual deles é mais sensível à priori? Repita o cálculo do LPD com as prioris mais difusas do item 2(g) e comente.

In [ ]:
cmp = pd.DataFrame({"log p(y|M) (BF)": LOGML, "rank BF": (-LOGML).rank().astype(int),
                    "LPD (LOO)": crit["LPD"], "rank LPD": crit["rank LPD"]})
display(cmp.round(3))
for a, b in [("M6", "M3"), ("M3", "M1"), ("M3", "M2"), ("M5", "M2")]:
    print(f"{a} vs {b}: log B = {LOGML[a] - LOGML[b]:7.3f}   log pseudo-B = LPD diff = "
          f"{crit.LPD[a] - crit.LPD[b]:7.3f}   (pseudo-B = {np.exp(crit.LPD[a] - crit.LPD[b]):.2f})")

In [ ]:
# Sampling uncertainty of an LPD difference (Vehtari et al. 2017): s.e. = sqrt(n) * sd(d_i)
for a, b in [("M6", "M3"), ("M3", "M1"), ("M3", "M2"), ("M5", "M4")]:
    d = loo[a].lpd - loo[b].lpd
    print(f"LPD({a}) - LPD({b}) = {d.sum():6.3f}, s.e. = {np.sqrt(n) * d.std(ddof=1):.3f}")

In [ ]:
loo_d = run_loo(prior_scale=SCALE, seed0=SEED + 800)
crit_d = criteria(loo_d)
sens = pd.DataFrame({"LPD, base": crit["LPD"], "LPD, diffuse": crit_d["LPD"],
                     "change LPD": crit_d["LPD"] - crit["LPD"],
                     "change log p(y|M)": comp["change"]})
print(f"elapsed {time.time() - T0:.0f} s")
sens.round(3)

### Answer 3(d)

**Result:** cross-validation and the Bayes factor reach the same two conclusions:
- Both regressors are needed.
- Student-t and Gaussian errors are tied: $\log B_{63}=0.036$ and the log pseudo-Bayes factor is $0.068$ (pseudo-$B$ = 1.07).

They differ only in the order of the weak one-regressor models: M2 is ahead of M1 by $p(y\mid M)$ and behind it by LPD. The Bayes factor is the prior-sensitive one. Under the diffuse priors of 2(g), $\log p(y\mid M)$ falls by 1.8 to 2.6, while the LPD changes by −0.17 to +0.03 and keeps its ranking.

**Pseudo-Bayes factor.** $\exp(\text{LPD}_k)=\prod_i\text{CPO}_i$ with $\text{CPO}_i=p(y_i\mid y_{(-i)},M_k)$, and $\exp(\text{LPD}_k-\text{LPD}_l)$ is the Geisser & Eddy (1979) pseudo-Bayes factor. Log Bayes factor and log pseudo-Bayes factor for each pair:

| pair | $\log B$ | log pseudo-$B$ | pseudo-$B$ |
|---|---|---|---|
| M3 vs M1 | 3.41 | 3.83 | 46 |
| M3 vs M2 | 2.96 | 3.96 | 53 |
| M5 vs M2 | 0.74 | 0.88 | – |
| M6 vs M3 | 0.04 | 0.07 | 1.07 |

**Why the two can disagree.** By the chain rule, $p(y\mid M)=p(y_1\mid M)\,p(y_2\mid y_1,M)\cdots p(y_n\mid y_{1:n-1},M)$. It is a sequential predictive score whose first factors are predictions made from the prior alone. The Bayes factor therefore judges the model together with its prior: which model, with its prior, predicted these data from scratch. The LPD scores every $y_i$ with a posterior based on the other 39 observations, where the data dominate the prior: which fitted model predicts a new observation best. The two questions coincide when the prior is well calibrated, and differ when it is vague or misplaced. The two methods also differ in their goals: the Bayes factor is consistent for selecting the true model when it is in the set, while LOO targets predictive accuracy and is more tolerant of an unnecessary parameter.

**Prior sensitivity.** The table above compares the LPD under the base and the diffuse priors. The LPD changes are of the order of a few Monte Carlo s.e. (largest −0.17, for M3), against −1.8 to −2.6 for $\log p(y\mid M)$. Under the diffuse priors the LPD order is still M6, M3, M5, M1, then M4 and M2 tied at −124.38. A wider prior costs the marginal likelihood a lot, because the prior predictive spreads its mass. It costs the LOO predictive almost nothing, because 39 observations wash it out.

**Sampling uncertainty.** The paired standard error $\sqrt n\,\operatorname{sd}(d_i)$ of each LPD difference (Vehtari, Gelman & Gabry 2017) is large with $n=40$: M6 − M3 = 0.068 (s.e. 0.41), M3 − M1 = 3.83 (3.56), M3 − M2 = 3.96 (2.75). By LOO standards, even the case for the second regressor is weak (at most 1.4 s.e.). This matches the "positive to strong" Bayes factors of 2(f) and reflects how little 40 observations can say.

#### Statement 3(e)

> **(e)** (Para os mais energizados) Em vez de reajustar o modelo 40 vezes, aproxime $p(y_i\mid y_{(-i)},M_k)$ a partir de um único ajuste com todos os dados, usando a identidade $p(y_i\mid y_{(-i)})=\{E[1/p(y_i\mid\theta)\mid y]\}^{-1}$ (Gelfand, Dey & Chang, 1992), ou o PSIS-LOO do pacote `loo` (Vehtari, Gelman & Gabry, 2017). Compare com os valores exatos do item (b) e comente as diferenças observação a observação.

### Answer 3(e): method

**Gelfand, Dey & Chang (1992).** Given $\theta$, $y_i$ is independent of $y_{(-i)}$, so
$$E\!\left[\frac{1}{p(y_i\mid\theta)}\,\Big|\,y\right]=\int\frac{p(y\mid\theta)p(\theta)}{p(y_i\mid\theta)\,p(y)}d\theta=\frac{\int p(y_{(-i)}\mid\theta)p(\theta)d\theta}{p(y)}=\frac{p(y_{(-i)})}{p(y)}=\frac{1}{p(y_i\mid y_{(-i)})}.$$
Hence $\widehat{\text{CPO}}_i=\left[\frac1S\sum_s 1/p(y_i\mid\theta^{(s)})\right]^{-1}$, using the $S=40{,}000$ draws of the full-data fit. This is importance sampling from the full posterior to the leave-one-out posterior, with weights $r_s\propto1/p(y_i\mid\theta^{(s)})$. It is a harmonic mean, but of a single observation's likelihood. Its weights have finite variance unless observation $i$ is very influential.

**PSIS-LOO (Vehtari, Gelman & Gabry 2017).** Use the same raw weights $r_s$. Fit a generalized Pareto distribution to the largest $M=\min(0.2S,3\sqrt S)=600$ of them (Zhang & Stephens estimator), replace those weights by the fitted quantiles, truncate at the largest raw weight, and normalize. Then $\log p(y_i\mid y_{(-i)})\approx\log\sum_s\tilde w_s\,p(y_i\mid\theta^{(s)})$. The Pareto shape $\hat k$ is the diagnostic: $\hat k<0.7$ means the estimate is reliable.

In [ ]:
def loglik_matrix(fit):
    """log p(y_i | theta^(g)) for a full-data fit, pooled over chains: (S*C, n)."""
    X, _ = design(fit["model"])
    mu = np.einsum("sck,ik->sci", fit["beta"], X).reshape(-1, n)
    sd = np.sqrt(fit["sigma2"]).reshape(-1, 1)
    if fit["nu"] is None:
        return stats.norm.logpdf(y, mu, sd)
    nu = fit["nu"].reshape(-1, 1)
    return log_t((y - mu) / sd, nu) - np.log(sd)


def gdc_loo(ll):
    """Gelfand-Dey-Chang: log CPO_i = -log E[1/p(y_i|theta) | y]."""
    return -(logsumexp(-ll, axis=0) - np.log(ll.shape[0]))


def gpdfit(x):
    """Generalized Pareto fit (Zhang & Stephens 2009, with the weak prior used in PSIS)."""
    N = len(x)
    m = 30 + int(np.sqrt(N))
    bs = 1 - np.sqrt(m / (np.arange(1, m + 1) - 0.5))
    bs = bs / (3 * x[int(N / 4 + 0.5) - 1]) + 1 / x[-1]
    ks = np.log1p(-bs[:, None] * x).mean(1)
    L = N * (np.log(-bs / ks) - ks - 1)
    w = 1 / np.exp(L[None, :] - L[:, None]).sum(1)   # w_i = 1 / sum_j exp(L_j - L_i)
    b = (bs * w).sum() / w.sum()
    k = np.log1p(-b * x).mean()
    sigma = -k / b
    k = (N * k + 10 * 0.5) / (N + 10)                 # weak prior shrinking k toward 0.5
    return k, sigma


def psis_loo(ll):
    """PSIS-LOO (Vehtari, Gelman & Gabry 2017). ll: (S, n). Returns (elpd_i, khat_i)."""
    S = ll.shape[0]
    M = int(np.ceil(min(0.2 * S, 3 * np.sqrt(S))))
    elpd, khat = np.empty(ll.shape[1]), np.empty(ll.shape[1])
    for i in range(ll.shape[1]):
        lw = -ll[:, i] - (-ll[:, i]).max()
        order = np.argsort(lw)
        cut = lw[order[-M - 1]]
        x = np.exp(lw[order[-M:]]) - np.exp(cut)
        k, sig = gpdfit(x)
        p = (np.arange(M) + 0.5) / M
        q = sig * np.expm1(-k * np.log1p(-p)) / k + np.exp(cut)
        lw[order[-M:]] = np.log(q)
        lw = np.minimum(lw, 0.0)
        lw -= logsumexp(lw)
        elpd[i], khat[i] = logsumexp(lw + ll[:, i]), k
    return elpd, khat

In [ ]:
appr, khat, pw = {}, {}, {}
for m in MODELS:
    ll = loglik_matrix(full[m])          # 40,000 draws x 40 observations, full-data fit
    g = gdc_loo(ll)
    p_, k_ = psis_loo(ll)
    pw[m] = pd.DataFrame({"exact": loo[m].lpd.values, "GDC": g, "PSIS": p_, "khat": k_}, index=idx)
    appr[m] = {"LPD exact": loo[m].lpd.sum(), "LPD GDC": g.sum(), "LPD PSIS": p_.sum(),
               "max |GDC - exact|_i": np.abs(g - loo[m].lpd).max(),
               "max |PSIS - exact|_i": np.abs(p_ - loo[m].lpd).max(), "max khat": k_.max()}
pd.DataFrame(appr).T.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for a, m in zip(ax, ["M3", "M6"]):
    d = pw[m]
    a.plot(idx, d.GDC - d.exact, "o", ms=4, label="GDC - exact")
    a.plot(idx, d.PSIS - d.exact, "x", ms=5, label="PSIS - exact")
    a.axhline(0, color="k", lw=0.8)
    a.set_title(m); a.set_xlabel("observation i"); a.set_ylabel("difference in log p(y_i | y_(-i))")
    a.legend(fontsize=8)
fig.suptitle("Figure. Single-fit approximations minus exact LOO, observation by observation")
fig.tight_layout(); plt.show()
w = pw["M6"].assign(d=lambda t: (t.PSIS - t.exact).abs()).sort_values("d", ascending=False).head(4)
w.round(3)

### Answer 3(e)

**Result:** both single-fit approximations reproduce the exact LOO. The totals differ from the exact LPD by 0.005 to 0.049, roughly the Monte Carlo s.e. of the exact values (0.025 to 0.034). Per observation the differences are at most 0.026, and every $\hat k$ is ≤ 0.53. Cost: 6 fits instead of 240.

- **GDC and PSIS are almost identical** (totals within 0.001). All $\hat k$ are below 0.7, and most below 0.5. The raw importance weights are already well behaved, so the Pareto smoothing has little to correct. No observation is influential enough to make the leave-one-out posterior far from the full posterior.
- **Where the differences are largest:** observations 40, 39, 5 and 9 in M6 (0.014 to 0.022), and 40, 9, 31 and 5 in M3 (Figure). These observations have large residuals (40, 39, 5, 9) or high leverage (31, life = 95): removing them moves the posterior most, so their weights $1/p(y_i\mid\theta)$ vary most. Observation 40 has the highest $\hat k$ in M6 (0.41). The approximation errors carry no systematic sign.
- **Same conclusions as the exact computation:** M6 ≈ M3 ≫ M5 > the other one-regressor models. With $\hat k$ this low, exact refitting was not necessary here. It would be needed for observations with $\hat k>0.7$.

---
## Computational details

In [ ]:
ess_all = pd.concat({m: summarize(full[m])[["ESS", "Rhat"]] for m in MODELS})
print(f"Seed: SEED = {SEED} (each call uses SEED + offset)")
print(f"Full-data fits: {N_CHAINS} chains, burn-in {N_BURN}, {N_KEEP} kept per chain "
      f"({N_CHAINS * N_KEEP} draws), no thinning")
print(f"LOO fits: 240 (x2 priors), burn-in {LOO_BURN}, {LOO_KEEP} kept each")
print(f"ESS range over all parameters: {ess_all.ESS.min():.0f} to {ess_all.ESS.max():.0f}; "
      f"max split R-hat {ess_all.Rhat.max():.4f}")
print(f"Total run time: {time.time() - T0:.0f} s")

### Conclusion

- **Data (Q1).** Impairment rises with life events ($r=0.37$) and falls with SES ($r=-0.40$). Both correlations are significant at 5%, and the two regressors are nearly uncorrelated ($r=0.12$, VIF 1.015).
- **Coefficients.** In the model with both variables, one s.d. more life events adds about 2.2 points of impairment and one s.d. more SES removes about 2.3 points, with 95% credible intervals that exclude zero.
- **Regressors.** Both regressors are needed: Bayes factors of 10 to 32 for the two-regressor models, $P(M_3\cup M_6\mid y)=0.90$, and a 16% lower LOO MSE.
- **Error distribution.** Gaussian and Student-t errors cannot be told apart: $B_{63}=1.04$, pseudo-$B=1.07$, and the posterior of $\nu$ almost equals its prior. M3, the simpler model, is the sensible final choice.
- **Bayes factors vs LOO.** The Bayes factors depend on the prior (tripling the prior s.d. moves the evidence for the second regressor from strong to positive, the Lindley–Bartlett effect). The LOO criteria barely move. With $n=40$, both approaches carry large uncertainty about small differences.